# RIDE IT – Drivers Engagement Analysis
## Capstone Project 1

**Objective:** Analyze driver activity and engagement, define key engagement metrics, identify high- and low-performing segments, and generate business insights for the Supply Product team.

**Datasets:** `rideit_drivers.csv` and `rideit_drivers_activity.csv`

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


## 2. Load the Datasets

In [ ]:
drivers = pd.read_csv('rideit_drivers.csv')
activity = pd.read_csv('rideit_drivers_activity.csv')

print('Drivers shape:', drivers.shape)
print('Activity shape:', activity.shape)
display(drivers.head())
display(activity.head())

## 3. Data Understanding and Quality Checks

In [ ]:
print('DRIVERS DATASET')
drivers.info()
print('\nMissing values:')
display(drivers.isnull().sum())
print('Exact duplicates:', drivers.duplicated().sum())
print('Unique driver IDs:', drivers['id_driver'].nunique())

print('\nACTIVITY DATASET')
activity.info()
print('\nMissing values:')
display(activity.isnull().sum())
print('Exact duplicates:', activity.duplicated().sum())

## 4. Data Cleaning
Convert date fields, standardize column names, and remove only exact duplicate rows. Missing ratings and Gold counts are preserved rather than imputed without evidence.

In [ ]:
drivers.columns = drivers.columns.str.strip().str.lower()
activity.columns = activity.columns.str.strip().str.lower()

drivers['date_registration'] = pd.to_datetime(drivers['date_registration'], errors='coerce')
activity['active_date'] = pd.to_datetime(activity['active_date'], errors='coerce')

drivers = drivers.drop_duplicates().copy()
activity = activity.drop_duplicates().copy()

print('Cleaned drivers shape:', drivers.shape)
print('Cleaned activity shape:', activity.shape)

## 5. Data Validation

In [ ]:
numeric_cols = ['offers','bookings','bookings_cancelled_by_passenger','bookings_cancelled_by_driver','rides']
print('Negative values by column:')
display((activity[numeric_cols] < 0).sum())

print('Rows where bookings > offers:', (activity['bookings'] > activity['offers']).sum())
print('Duplicate driver-date combinations:', activity.duplicated(['id_driver','active_date']).sum())
print('Activity drivers missing from driver profile:', (~activity['id_driver'].isin(drivers['id_driver'])).sum())

## 6. Create a Unique Driver Master Table
Some drivers appear under multiple service types. Their service types are combined to prevent double-counting at driver level.

In [ ]:
profile_cols = ['id_driver','date_registration','driver_rating','gold_level_count','receive_marketing','country_code']

def combine_services(series):
    values = sorted(set(series.dropna().astype(str)))
    return ' / '.join(values) if values else np.nan

driver_master = (drivers.groupby(profile_cols, dropna=False, as_index=False)
                 .agg(service_type=('service_type', combine_services)))

print('Unique driver master rows:', len(driver_master))
display(driver_master.head())

## 7. Aggregate Driver Activity

In [ ]:
driver_activity = activity.groupby('id_driver', as_index=False).agg(
    active_days=('active_date','nunique'),
    first_active_date=('active_date','min'),
    last_active_date=('active_date','max'),
    total_offers=('offers','sum'),
    total_bookings=('bookings','sum'),
    passenger_cancellations=('bookings_cancelled_by_passenger','sum'),
    driver_cancellations=('bookings_cancelled_by_driver','sum'),
    total_rides=('rides','sum')
)

display(driver_activity.head())

## 8. Feature Engineering and Engagement Metrics
- **Active Days:** frequency of platform activity
- **Rides per Active Day:** engagement intensity/productivity
- **Driver Cancellation Rate:** driver cancellations divided by bookings
- **Ride Completion Rate:** completed rides divided by bookings
- **Tenure Days:** time from registration to the latest activity date in the dataset

In [ ]:
analysis = driver_master.merge(driver_activity, on='id_driver', how='left')

analysis['rides_per_active_day'] = (analysis['total_rides'] / analysis['active_days']).where(analysis['active_days'] > 0)
analysis['driver_cancellation_rate'] = (analysis['driver_cancellations'] / analysis['total_bookings']).where(analysis['total_bookings'] > 0)
analysis['ride_completion_rate'] = (analysis['total_rides'] / analysis['total_bookings']).where(analysis['total_bookings'] > 0)

latest_date = activity['active_date'].max()
analysis['tenure_days'] = (latest_date - analysis['date_registration']).dt.days

display(analysis.head())

## 9. Overall KPI Analysis

In [ ]:
overall_kpis = pd.Series({
    'Unique Drivers': analysis['id_driver'].nunique(),
    'Total Active Driver-Days': analysis['active_days'].sum(),
    'Total Offers': analysis['total_offers'].sum(),
    'Total Bookings': analysis['total_bookings'].sum(),
    'Total Completed Rides': analysis['total_rides'].sum(),
    'Rides per Active Day': analysis['total_rides'].sum() / analysis['active_days'].sum(),
    'Driver Cancellation Rate (%)': analysis['driver_cancellations'].sum() / analysis['total_bookings'].sum() * 100,
    'Ride Completion Rate (%)': analysis['total_rides'].sum() / analysis['total_bookings'].sum() * 100
})
display(overall_kpis.to_frame('Value'))

## 10. Monthly Engagement Trend

In [ ]:
activity['month'] = activity['active_date'].dt.to_period('M').astype(str)
monthly = activity.groupby('month').agg(
    active_drivers=('id_driver','nunique'),
    active_driver_days=('id_driver','size'),
    total_rides=('rides','sum'),
    total_bookings=('bookings','sum'),
    driver_cancellations=('bookings_cancelled_by_driver','sum')
).reset_index()

monthly['rides_per_active_day'] = monthly['total_rides'] / monthly['active_driver_days']
monthly['rides_per_active_driver'] = monthly['total_rides'] / monthly['active_drivers']
monthly['driver_cancellation_rate_pct'] = monthly['driver_cancellations'] / monthly['total_bookings'] * 100
display(monthly)

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly['month'], monthly['active_drivers'], marker='o')
plt.title('Monthly Active Drivers')
plt.xlabel('Month')
plt.ylabel('Active Drivers')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(monthly['month'], monthly['rides_per_active_day'], marker='o')
plt.title('Monthly Rides per Active Day')
plt.xlabel('Month')
plt.ylabel('Rides per Active Day')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. Segment Analysis
Compare engagement across country, service type, marketing preference, Gold achievement, and driver rating.

In [ ]:
def segment_analysis(column):
    return analysis.groupby(column, dropna=False).agg(
        drivers=('id_driver','nunique'),
        avg_active_days=('active_days','mean'),
        total_rides=('total_rides','sum'),
        avg_rides_per_active_day=('rides_per_active_day','mean'),
        avg_driver_cancellation_rate=('driver_cancellation_rate','mean')
    ).reset_index()

country_analysis = segment_analysis('country_code')
service_analysis = segment_analysis('service_type')
marketing_analysis = segment_analysis('receive_marketing')

display(country_analysis.sort_values('total_rides', ascending=False))
display(service_analysis)
display(marketing_analysis)

## 12. Gold-Level Segmentation

In [ ]:
analysis['gold_segment'] = pd.cut(
    analysis['gold_level_count'], [-1,0,5,20,np.inf],
    labels=['No Gold','1-5','6-20','21+'])
analysis['gold_segment'] = analysis['gold_segment'].astype(object)
analysis.loc[analysis['gold_level_count'].isna(),'gold_segment'] = 'Unknown'

gold_analysis = segment_analysis('gold_segment')
display(gold_analysis)

## 13. Driver Rating Segmentation

In [ ]:
analysis['rating_segment'] = pd.cut(
    analysis['driver_rating'], [0,4.0,4.5,4.8,5.0],
    labels=['<=4.0','4.01-4.5','4.51-4.8','4.81-5.0'], include_lowest=True)
analysis['rating_segment'] = analysis['rating_segment'].astype(object)
analysis.loc[analysis['driver_rating'].isna(),'rating_segment'] = 'Unknown'

rating_analysis = segment_analysis('rating_segment')
display(rating_analysis)

## 14. Export Prepared Data
Save cleaned datasets and the driver-level analytical dataset for SQL, Power BI, and stakeholder reporting.

In [ ]:
driver_master.to_csv('rideit_drivers_cleaned.csv', index=False)
activity.to_csv('rideit_drivers_activity_cleaned.csv', index=False)
analysis.to_csv('rideit_driver_analysis.csv', index=False)

print('Files exported successfully.')

## 15. Conclusion and Business Recommendations

Use the calculated KPI trends and segment tables above to summarize:

1. Whether driver engagement is improving or declining over time.
2. Which countries and service types show stronger engagement.
3. Whether marketing opt-in is associated with engagement.
4. How Gold achievement and driver ratings relate to activity.
5. Which driver segments should be targeted with retention, incentive, communication, or product initiatives.

> Final conclusions should be based on the outputs generated when the notebook is executed.